|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The GPU<h1>|
|<h2>Lecture:</h2>|<h1><b>Code challenge: predict, then measure<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# Find the repo root. The directory you start from does not matter.
import sys
import time
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib
import copy
from transformers import AutoModelForCausalLM

Predict first. Then measure.

You measured two numbers of your card: its bandwidth and its FLOP/s. With them
you can predict the time of any work before you run it. It is the time to read
its bytes, or the time to do its FLOPs: the larger of the two. This challenge makes
the prediction for four kinds of work, and then checks it.

In [ ]:
### run this cell
PEAK_BANDWIDTH = cudalib.peak_bandwidth() * 1e9     # bytes/s, measured on this card
PEAK_FLOPS = cudalib.matmul_flops()                  # FLOP/s of a bf16 matmul, measured
print(f'memory bandwidth: {PEAK_BANDWIDTH/1e9:6.0f} GB/s')
print(f'arithmetic:       {PEAK_FLOPS/1e12:6.1f} TFLOP/s')

# Exercise 1: the prediction

The work cannot be faster than its bytes at full bandwidth. It cannot be
faster than its FLOPs at full arithmetic speed. It waits for the slower of
the two. Return the prediction in milliseconds.

In [ ]:
def predict_ms(flops, num_bytes):
  """The shortest possible time for this work on this card, in ms."""
  return 1e3 * max(flops / PEAK_FLOPS, num_bytes / PEAK_BANDWIDTH)

assert predict_ms(0, PEAK_BANDWIDTH) == 1e3        # one second of bytes
assert predict_ms(PEAK_FLOPS, 0) == 1e3            # one second of arithmetic
print('the prediction is correct')

# Exercise 2: count the FLOPs and the bytes

Fill in the FLOPs and the bytes of each workload. Count only the data that
must come from memory: the inputs one time, and the output one time.

- the vector add of fp32 numbers, 4 bytes each.
- a GEMV: a (SIZE x SIZE) bf16 matrix times a vector. A multiply and an add
  are 2 FLOPs. The vector is small: ignore it.
- a matmul: two (SIZE x SIZE) bf16 matrices, and a result of the same size.

Then compare the prediction with the measurement. The ratio is the fraction
of the best possible speed that the work reaches.

The peak numbers are also measurements, from simple kernels. A tuned library
kernel can be some percent faster than them. So a ratio a little above 100% is
not a fault.

In [ ]:
NUM_VALUES, SIZE = 2**26, 8192     # all larger than the L2 cache
a, b = torch.rand(NUM_VALUES, device='cuda'), torch.rand(NUM_VALUES, device='cuda')
matrix = torch.randn(SIZE, SIZE, device='cuda', dtype=torch.bfloat16)
vector = torch.randn(SIZE, 1, device='cuda', dtype=torch.bfloat16)

workloads = {
  #            (function,               FLOP,                   bytes)
  'vector add': (lambda: a + b,          NUM_VALUES            , 12 * NUM_VALUES),
  'GEMV':       (lambda: matrix @ vector, 2 * SIZE * SIZE       , 2 * SIZE * SIZE),
  'matmul':     (lambda: matrix @ matrix, 2 * SIZE**3           , 3 * 2 * SIZE * SIZE),
}

print(f"{'work':<11} {'predicted':>10} {'measured':>10} {'of the best':>12}")
for name, (function, flops, num_bytes) in workloads.items():
  predicted = predict_ms(flops, num_bytes)
  measured = cudalib.bench_ms(function, best_of=3)
  print(f'{name:<11} {predicted:>8.3f}ms {measured:>8.3f}ms {100*predicted/measured:>11.0f}%')

# Exercise 3: one decode step of a real model

At batch 1, one decode step reads all the weights of the model, and does about
2 FLOPs for each weight. Count the bytes of the weights, and predict the step
time. The cell then measures one real decode step with a KV cache.

In [ ]:
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', dtype=torch.bfloat16).cuda().eval()
weight_bytes = sum(weight.numel() * weight.element_size() for weight in model.parameters())
num_weights = sum(weight.numel() for weight in model.parameters())
predicted = predict_ms(2 * num_weights, weight_bytes)

with torch.no_grad():
  prompt = torch.randint(0, 1000, (1, 128), device='cuda')
  cache = model(prompt, use_cache=True).past_key_values
  next_token = torch.randint(0, 1000, (1, 1), device='cuda')
  measured = cudalib.bench_ms(
      lambda: model(next_token, past_key_values=copy.copy(cache), use_cache=True),
      iters=20, warmup=5, best_of=3)

print(f'weights: {weight_bytes/1e9:.2f} GB')
print(f'predicted step: {predicted:.2f} ms')
print(f'measured step:  {measured:.2f} ms')
print(f'the step reaches {100*predicted/measured:.0f}% of the best possible speed')

### Before you open the solution

1. The matmul in Exercise 2 is far from its prediction on some cards, and near
   it on others. What does the matmul wait for, that the formula ignores?
2. The decode step in Exercise 3 is much slower than its prediction. The
   prediction counts bytes and FLOPs. What else does a step cost? The last
   notebook measured one candidate.
3. The prediction says that a batch of 8 sequences costs almost the same time
   as a batch of 1. Why? Part 1 starts from this question.